# 03 - Explainability with SHAP

This notebook explains the final fraud detection model selected during modeling.

**Final modeling decision:**
- Model: Random Forest
- Feature version: Version B - Without ExtremeFlags
- Expected benchmark: PR-AUC around 0.816 and ROC-AUC around 0.986

The goal is to translate model behavior into feature-level explanations that can support analyst review and a future LLM-powered explanation layer.

## 1. Setup

This section imports the libraries used for modeling, evaluation, and SHAP explainability. The notebook keeps the same `random_state=42` used in the modeling workflow so results remain reproducible.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

try:
    from imblearn.over_sampling import SMOTE
except ImportError as exc:
    raise ImportError(
        "imbalanced-learn is required for SMOTE. Install it with: pip install imbalanced-learn"
    ) from exc

try:
    import shap
except ImportError as exc:
    raise ImportError("SHAP is required for this notebook. Install it with: pip install shap") from exc

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TEST_SIZE = 0.20
SHAP_SAMPLE_SIZE = 2000

DATA_PATH = Path("data/processed/creditcard_cleaned_engineered.csv")
SHAP_PLOTS_DIR = Path("outputs/plots/shap")
SHAP_IMPORTANCE_PATH = Path("outputs/shap_feature_importance.csv")

SHAP_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
SHAP_IMPORTANCE_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"SHAP plots will be saved to: {SHAP_PLOTS_DIR}")
print(f"SHAP feature importance table will be saved to: {SHAP_IMPORTANCE_PATH}")

**Recruiter-friendly takeaway:** This notebook is designed like a production analysis handoff: it defines paths, constants, and reproducible settings up front, then saves every explainability artifact to a predictable output folder.

## 2. Load the Processed Dataset

The processed dataset already includes cleaned transaction fields and engineered features from the earlier EDA and feature engineering workflow.

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Fraud rate:", round(df["Class"].mean() * 100, 4), "%")
display(df.head())

**Recruiter-friendly takeaway:** The explainability workflow starts from the same processed data used for modeling, which keeps the analysis consistent and avoids explaining a different feature space than the one used to make predictions.

## 3. Create Version B Features

Version B intentionally removes all `_ExtremeFlag` columns. This tests and explains the final model without explicit outlier indicator shortcuts, using the feature version selected in the modeling notebook.

In [ ]:
TARGET = "Class"
raw_or_intermediate_cols = ["Time", "Amount", "Hour", "HourOfDay"]
extreme_flag_cols = [col for col in df.columns if col.endswith("_ExtremeFlag")]

base_drop_cols = [TARGET] + [col for col in raw_or_intermediate_cols if col in df.columns]
feature_cols_A = [col for col in df.columns if col not in base_drop_cols]
feature_cols_B = [col for col in feature_cols_A if col not in extreme_flag_cols]

X = df[feature_cols_B].copy()
y = df[TARGET].copy()

print(f"ExtremeFlag columns removed: {len(extreme_flag_cols)}")
print(extreme_flag_cols)
print(f"Version B feature count: {len(feature_cols_B)}")
print(feature_cols_B)

assert not any(col.endswith("_ExtremeFlag") for col in X.columns), "Version B must exclude ExtremeFlag columns."

**Recruiter-friendly takeaway:** The model is being explained under the exact final feature decision: no `_ExtremeFlag` columns are allowed, so the explanations reflect signal learned from the underlying transaction variables and engineered time/amount features.

## 4. Stratified Train-Test Split

Fraud is rare, so the split is stratified to preserve the fraud rate in both train and test sets. The test set remains untouched and imbalanced.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train fraud rate:", round(y_train.mean() * 100, 4), "%")
print("Test fraud rate:", round(y_test.mean() * 100, 4), "%")

**Recruiter-friendly takeaway:** Stratification protects the evaluation from accidental class imbalance shifts, which is especially important when the positive class is extremely rare.

## 5. Apply SMOTE to Training Data Only

SMOTE is applied only after the split and only to `X_train`. This prevents synthetic fraud examples from leaking into the test set.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", X_train.shape, " Fraud rate:", round(y_train.mean() * 100, 4), "%")
print("After SMOTE: ", X_train_resampled.shape, " Fraud rate:", round(y_train_resampled.mean() * 100, 2), "%")

**Recruiter-friendly takeaway:** This is the correct imbalance workflow: the model gets help learning minority-class patterns during training, while evaluation remains honest on real, imbalanced test data.

## 6. Train the Final Random Forest Model

This reproduces the final model decision from the modeling notebook. The trained model stays in memory for explainability and is not saved to disk.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=250,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_model.fit(X_train_resampled, y_train_resampled)
print("Random Forest training complete.")

**Recruiter-friendly takeaway:** The explainability analysis uses the same final model family and feature version selected by performance, making the SHAP results directly connected to the project decision.

## 7. Brief Evaluation Check

Before explaining the model, we confirm that this reproduction lands close to the modeling notebook benchmark: PR-AUC around 0.816 and ROC-AUC around 0.986.

In [ ]:
y_score = rf_model.predict_proba(X_test)[:, 1]
y_pred = (y_score >= 0.50).astype(int)

metrics = {
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_score),
    "pr_auc_avg_precision": average_precision_score(y_test, y_score),
}

metrics_df = pd.DataFrame([metrics])
display(metrics_df)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Legitimate", "Fraud"],
    yticklabels=["Legitimate", "Fraud"],
)
plt.title("Random Forest Confusion Matrix - Version B")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"], zero_division=0))

**Recruiter-friendly takeaway:** The brief evaluation confirms that the explainability work is attached to a strong fraud model rather than a toy example. PR-AUC is emphasized because it is more meaningful than accuracy when fraud cases are rare.

## 8. Compute SHAP Values on a Test Sample

Tree SHAP explains how each feature pushes the Random Forest prediction higher or lower. To keep runtime practical, this notebook samples 2,000 rows from the test set by default.

In [ ]:
def get_positive_class_shap_values(shap_values):
    """Return SHAP values for the fraud class across older and newer SHAP APIs."""
    if isinstance(shap_values, list):
        return shap_values[1]
    values = np.asarray(shap_values)
    if values.ndim == 3:
        return values[:, :, 1]
    return values


def get_positive_expected_value(expected_value):
    values = np.asarray(expected_value)
    if values.ndim > 0 and len(values) > 1:
        return values[1]
    return float(values)


sample_n = min(SHAP_SAMPLE_SIZE, len(X_test))
X_shap = X_test.sample(n=sample_n, random_state=RANDOM_STATE)
y_shap = y_test.loc[X_shap.index]
y_score_shap = pd.Series(y_score, index=X_test.index).loc[X_shap.index]

explainer = shap.TreeExplainer(rf_model)
raw_shap_values = explainer.shap_values(X_shap)
shap_values_fraud = get_positive_class_shap_values(raw_shap_values)
expected_value_fraud = get_positive_expected_value(explainer.expected_value)

print("SHAP sample shape:", X_shap.shape)
print("SHAP values shape:", shap_values_fraud.shape)
print("Expected fraud-class model output:", expected_value_fraud)

**Recruiter-friendly takeaway:** Sampling makes the explainability step fast enough for an applied workflow while still using real test-set examples. The SHAP values show which features most influence the model's fraud probability.

## 9. Global SHAP Feature Importance

Global SHAP importance ranks features by their average absolute impact on fraud predictions across the sampled test rows.

In [ ]:
shap_importance = (
    pd.DataFrame({
        "feature": X_shap.columns,
        "mean_abs_shap": np.abs(shap_values_fraud).mean(axis=0),
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

shap_importance.to_csv(SHAP_IMPORTANCE_PATH, index=False)
display(shap_importance.head(20))

plt.figure(figsize=(9, 6))
sns.barplot(
    data=shap_importance.head(20),
    x="mean_abs_shap",
    y="feature",
    color="#3b7ddd",
)
plt.title("Top 20 Global SHAP Feature Importances - Fraud Class")
plt.xlabel("Mean absolute SHAP value")
plt.ylabel("Feature")
plt.tight_layout()
global_bar_path = SHAP_PLOTS_DIR / "global_shap_feature_importance_bar.png"
plt.savefig(global_bar_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved global SHAP feature importance plot to: {global_bar_path}")
print(f"Saved SHAP feature importance table to: {SHAP_IMPORTANCE_PATH}")

**Recruiter-friendly takeaway:** This table turns a complex Random Forest into a ranked list of drivers. It helps stakeholders quickly see which transformed transaction patterns matter most to fraud risk.

## 10. SHAP Summary Plot

The summary plot shows both importance and direction. Each dot is one transaction: red means a high feature value and blue means a low feature value.

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values_fraud,
    X_shap,
    max_display=20,
    show=False,
)
plt.title("SHAP Summary Plot - Random Forest Fraud Class")
plt.tight_layout()
summary_path = SHAP_PLOTS_DIR / "shap_summary_plot.png"
plt.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved SHAP summary plot to: {summary_path}")

**Recruiter-friendly takeaway:** The summary plot answers two business questions at once: which variables matter most, and whether high or low values of those variables tend to push a transaction toward fraud.

## 11. Local Explanations for High-Risk Fraud Predictions

This section selects fraud cases that the model scored as high risk and explains the strongest feature contributions for each transaction.

In [ ]:
test_results = X_test.copy()
test_results["actual"] = y_test
test_results["predicted"] = y_pred
test_results["fraud_probability"] = y_score

high_risk_frauds = (
    test_results[(test_results["actual"] == 1) & (test_results["predicted"] == 1)]
    .sort_values("fraud_probability", ascending=False)
    .head(3)
)

display(high_risk_frauds[["actual", "predicted", "fraud_probability"]])

for rank, idx in enumerate(high_risk_frauds.index, start=1):
    row = X_test.loc[[idx]]
    row_shap = get_positive_class_shap_values(explainer.shap_values(row))[0]
    local_df = (
        pd.DataFrame({
            "feature": X_test.columns,
            "feature_value": row.iloc[0].values,
            "shap_value": row_shap,
            "abs_shap_value": np.abs(row_shap),
        })
        .sort_values("abs_shap_value", ascending=False)
        .head(10)
    )

    print(f"High-risk true fraud #{rank} | index={idx} | fraud_probability={test_results.loc[idx, 'fraud_probability']:.4f}")
    display(local_df)

    plt.figure(figsize=(8, 5))
    colors = np.where(local_df["shap_value"] >= 0, "#d62728", "#1f77b4")
    plt.barh(local_df["feature"][::-1], local_df["shap_value"][::-1], color=colors[::-1])
    plt.axvline(0, color="black", linewidth=1)
    plt.title(f"Local SHAP Drivers - High-Risk Fraud #{rank}")
    plt.xlabel("SHAP value for fraud class")
    plt.tight_layout()
    local_path = SHAP_PLOTS_DIR / f"local_high_risk_fraud_{rank}_index_{idx}.png"
    plt.savefig(local_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved local high-risk fraud explanation to: {local_path}")

**Recruiter-friendly takeaway:** Local explanations make the model reviewable at the individual-transaction level. Instead of only saying "high risk," the workflow identifies the features that pushed that specific transaction toward a fraud decision.

## 12. Local Explanations for False Negatives

False negatives are actual fraud cases that the model missed at the default 0.50 threshold. These are important because they show where fraud patterns may be subtle, ambiguous, or below the decision cutoff.

In [ ]:
false_negatives = (
    test_results[(test_results["actual"] == 1) & (test_results["predicted"] == 0)]
    .sort_values("fraud_probability", ascending=False)
    .head(3)
)

display(false_negatives[["actual", "predicted", "fraud_probability"]])

if false_negatives.empty:
    print("No false negatives were found at the 0.50 threshold in this test split.")
else:
    for rank, idx in enumerate(false_negatives.index, start=1):
        row = X_test.loc[[idx]]
        row_shap = get_positive_class_shap_values(explainer.shap_values(row))[0]
        local_df = (
            pd.DataFrame({
                "feature": X_test.columns,
                "feature_value": row.iloc[0].values,
                "shap_value": row_shap,
                "abs_shap_value": np.abs(row_shap),
            })
            .sort_values("abs_shap_value", ascending=False)
            .head(10)
        )

        print(f"False negative #{rank} | index={idx} | fraud_probability={test_results.loc[idx, 'fraud_probability']:.4f}")
        print("Positive SHAP values pushed toward fraud; negative SHAP values pushed away from fraud.")
        display(local_df)

        plt.figure(figsize=(8, 5))
        colors = np.where(local_df["shap_value"] >= 0, "#d62728", "#1f77b4")
        plt.barh(local_df["feature"][::-1], local_df["shap_value"][::-1], color=colors[::-1])
        plt.axvline(0, color="black", linewidth=1)
        plt.title(f"Local SHAP Drivers - False Negative #{rank}")
        plt.xlabel("SHAP value for fraud class")
        plt.tight_layout()
        local_path = SHAP_PLOTS_DIR / f"local_false_negative_{rank}_index_{idx}.png"
        plt.savefig(local_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved false-negative explanation to: {local_path}")

**Recruiter-friendly takeaway:** False-negative explanations are especially useful for model improvement. They reveal which features lowered the fraud score, helping analysts understand missed cases and decide whether threshold tuning, new features, or additional review rules are needed.

## 13. How SHAP Connects to the Future LLM Fraud Explanation Layer

SHAP provides model-grounded reasons for each fraud prediction. It identifies which features pushed the Random Forest score higher or lower, so the explanation stays tied to the trained model's actual behavior.

The future LLM layer will not replace SHAP or invent new reasoning. Its role will be to convert these SHAP outputs into human-readable fraud investigation explanations that analysts can quickly understand and act on.

A practical LLM explanation layer could use:
- The model's fraud probability
- The top positive SHAP drivers that increased fraud risk
- The top negative SHAP drivers that reduced fraud risk
- Whether the case was a true positive, false positive, or false negative during evaluation
- Plain-language feature descriptions for fields such as `LogAmount`, `IsZeroAmount`, `HourSin`, and anonymized PCA components

The LLM should ground every explanation in the SHAP table for that transaction. For example, it can say that a case was flagged because several features strongly pushed the fraud score upward, while also noting any features that reduced confidence.

This creates a bridge between predictive modeling and analyst-facing decision support: the Random Forest produces the risk score, SHAP produces the evidence, and the LLM translates that evidence into a concise explanation for fraud operations.